# Gold: lab_test_trends

Builds `dbw_caresync_dev.gold.lab_test_trends` (one row per test type per calendar month) from `silver.lab_results`, per STTM `sttm_silver_to_gold.xlsx` (sheet `lab_test_trends`).

* **Driving table:** `silver.lab_results` grouped by `test_name`, `year(test_date)`, `month(test_date)` (no filters)
* **Joins (LEFT, optional):** `silver.calendar` on `date_key = date_format(date_trunc('month', test_date), 'yyyyMMdd')`
* **Window:** `LAG(abnormal_rate_pct)` partitioned by `test_name`, ordered by `year, month`
* **Load:** FULL — `CREATE OR REPLACE TABLE ... AS SELECT`
* **Schema note:** `silver.calendar.date_key` is `int`, so the formatted month-start string is cast to `INT` for the join.

In [0]:
%sql
-- FULL rebuild: one row per test_name per calendar month (driving table silver.lab_results)
CREATE OR REPLACE TABLE dbw_caresync_dev.gold.lab_test_trends AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                                    AS year,
    month(test_date)                                   AS month,
    CAST(date_trunc('MONTH', MIN(test_date)) AS DATE)  AS month_start_date,
    COUNT(*)                                           AS total_tests,
    COUNT_IF(result_status = 'Abnormal')               AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')               AS critical_test_count
  FROM dbw_caresync_dev.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    *,
    CAST(abnormal_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
trend AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  t.test_name,
  t.year,
  t.month,
  c.month_name,
  c.quarter_name,
  CAST(t.total_tests AS INT)                                             AS total_tests,
  CAST(t.abnormal_test_count AS INT)                                     AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                     AS critical_test_count,
  t.abnormal_rate_pct,
  t.critical_rate_pct,
  CAST(t.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                   AS prev_month_abnormal_rate_pct,
  CASE
    WHEN t.prev_month_abnormal_rate_pct IS NULL THEN NULL
    ELSE CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2))
  END                                                                    AS abnormal_rate_change_pct
FROM trend t
LEFT JOIN dbw_caresync_dev.silver.calendar c
  ON c.date_key = CAST(date_format(t.month_start_date, 'yyyyMMdd') AS INT)

In [0]:
# Runs only after the CREATE OR REPLACE TABLE cell above succeeds; returns the Gold row count to ADF (runOutput)
target_table = "dbw_caresync_dev.gold.lab_test_trends"
record_count = spark.table(target_table).count()
print(f"SUCCESS: {record_count} record(s) in {target_table}")
dbutils.notebook.exit(str(record_count))